In [3]:
import pygame
import numpy as np
from ai2thor.controller import Controller

# AI2-THOR 컨트롤러 초기화 (Unity 창 크기 설정 및 제3자 카메라 렌더링 활성화)
width = 800
height = 600

controller = Controller(
    scene="FloorPlan1",
    width=width,
    height=height,
    renderThirdPartyCameras=True,
    fieldOfView=90,
)

# 초기 위치와 회전 정보를 저장할 변수
initial_position = None
initial_rotation = None

# Pygame 초기화
pygame.init()
screen = pygame.display.set_mode((width, height))
pygame.display.set_caption("AI2-THOR Control")


# 프로그램 시작 시 에이전트의 현재 위치와 회전을 저장
def save_initial_position():
    global initial_position, initial_rotation
    agent_metadata = controller.last_event.metadata["agent"]
    initial_position = agent_metadata["position"]
    initial_rotation = agent_metadata["rotation"]
    print("초기 위치와 회전이 저장되었습니다.")


# 에이전트 초기 위치로 텔레포트하는 함수
def teleport_to_initial_position():
    global initial_position, initial_rotation
    if initial_position and initial_rotation:
        event = controller.step(
            action="Teleport",
            position=initial_position,
            rotation=initial_rotation,
            horizon=controller.last_event.metadata["agent"]["cameraHorizon"],
            standing=controller.last_event.metadata["agent"]["isStanding"],
            forceAction=True,
        )
        if event.metadata["lastActionSuccess"]:
            print("초기 위치로 텔레포트되었습니다.")
            # 현재 카메라 모드에 따라 카메라 업데이트
            if camera_mode == "third_person":
                update_third_person_camera()
            elif camera_mode == "top_down":
                update_top_down_camera()
        else:
            print("초기 위치로 이동할 수 없습니다.")
    else:
        print("초기 위치가 저장되지 않았습니다.")


# 초기 위치와 회전 저장
save_initial_position()

# 카메라와 에이전트 움직임 속도 설정
move_step = 0.25  # 에이전트 이동 거리
rotate_step = 15  # 카메라 회전 각도
run_speed_multiplier = 2  # Shift 키 사용 시 이동 속도 증가

camera_mode = "egocentric"  # 초기값을 egocentric로 설정
camera_modes = ["egocentric", "third_person", "top_down"]  # 카메라 모드 리스트


# 뷰 토글 함수
def toggle_view():
    global camera_mode
    current_index = camera_modes.index(camera_mode)
    next_index = (current_index + 1) % len(camera_modes)
    camera_mode = camera_modes[next_index]

    if camera_mode == "egocentric":
        set_egocentric_view()
    elif camera_mode == "third_person":
        set_third_person_view()
    elif camera_mode == "top_down":
        set_top_down_view()


# 3인칭 뷰 설정 함수
def set_third_person_view():
    agent_position = controller.last_event.metadata["agent"]["position"]
    agent_rotation = controller.last_event.metadata["agent"]["rotation"]

    # 카메라 위치 설정: 에이전트 뒤쪽으로 일정 거리 떨어진 곳에 배치
    distance_behind = 1.0  # 에이전트 뒤로 1.0 단위만큼 이동
    height_above = 1.5  # 에이전트 위로 이동

    # 방향에 따라 카메라 위치 계산
    theta = np.deg2rad(agent_rotation["y"])
    offset_x = -distance_behind * np.sin(theta)
    offset_z = -distance_behind * np.cos(theta)

    third_person_camera_position = {
        "x": agent_position["x"] + offset_x,
        "y": agent_position["y"] + height_above,
        "z": agent_position["z"] + offset_z,
    }

    # 카메라가 에이전트를 바라보도록 설정
    camera_rotation = {
        "x": 20,  # 카메라를 아래로 20도 기울임
        "y": agent_rotation["y"],
        "z": 0,
    }

    controller.step(
        action="AddThirdPartyCamera",
        position=third_person_camera_position,
        rotation=camera_rotation,
        fieldOfView=60,
    )


# 3인칭 카메라 업데이트 함수
def update_third_person_camera():
    agent_position = controller.last_event.metadata["agent"]["position"]
    agent_rotation = controller.last_event.metadata["agent"]["rotation"]

    distance_behind = 1.0
    height_above = 1.5

    theta = np.deg2rad(agent_rotation["y"])
    offset_x = -distance_behind * np.sin(theta)
    offset_z = -distance_behind * np.cos(theta)

    third_person_camera_position = {
        "x": agent_position["x"] + offset_x,
        "y": agent_position["y"] + height_above,
        "z": agent_position["z"] + offset_z,
    }

    camera_rotation = {
        "x": 20,  # 아래로 기울기 유지
        "y": agent_rotation["y"],
        "z": 0,
    }

    controller.step(
        action="UpdateThirdPartyCamera",
        thirdPartyCameraId=0,
        position=third_person_camera_position,
        rotation=camera_rotation,
        fieldOfView=60,
    )


# 탑 뷰 설정 함수
def set_top_down_view():
    agent_position = controller.last_event.metadata["agent"]["position"]
    # 에이전트 바로 위에 카메라 위치 설정
    height_above = 5.0  # 필요에 따라 높이 조정

    top_down_camera_position = {
        "x": agent_position["x"],
        "y": agent_position["y"] + height_above,
        "z": agent_position["z"],
    }

    # 카메라가 바로 아래를 바라보도록 회전 설정
    camera_rotation = {
        "x": -90,  # 아래로 90도 회전하여 내려다봄 (x 축에 대해 -90도)
        "y": 0,  # y 회전을 0으로 설정
        "z": 0,
    }

    controller.step(
        action="AddThirdPartyCamera",
        position=top_down_camera_position,
        rotation=camera_rotation,
        fieldOfView=90,
    )


# 탑 뷰 카메라 업데이트 함수
def update_top_down_camera():
    agent_position = controller.last_event.metadata["agent"]["position"]

    height_above = 5.0

    top_down_camera_position = {
        "x": agent_position["x"],
        "y": agent_position["y"] + height_above,
        "z": agent_position["z"],
    }

    camera_rotation = {
        "x": -90,  # 아래로 90도 회전하여 내려다봄 (x 축에 대해 -90도)
        "y": 0,
        "z": 0,
    }

    controller.step(
        action="UpdateThirdPartyCamera",
        thirdPartyCameraId=0,
        position=top_down_camera_position,
        rotation=camera_rotation,
        fieldOfView=90,
    )


# 1인칭 뷰 설정 함수
def set_egocentric_view():
    # 제3자 카메라를 시야에서 제거
    controller.step(
        action="UpdateThirdPartyCamera",
        thirdPartyCameraId=0,
        position={"x": 0, "y": -100, "z": 0},
    )


# 에이전트의 이동 처리 함수
def move_agent(move_action, run_mode=False):
    speed = move_step * run_speed_multiplier if run_mode else move_step
    event = controller.step(action=move_action, moveMagnitude=speed)
    if not event.metadata["lastActionSuccess"]:
        print("이동이 막혔습니다. 원하는 위치로 이동할 수 없습니다.")
    else:
        if camera_mode == "third_person":
            update_third_person_camera()
        elif camera_mode == "top_down":
            update_top_down_camera()


running = True

while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

        # 키다운 이벤트 처리
        elif event.type == pygame.KEYDOWN:
            if event.key == pygame.K_v:
                toggle_view()
            if event.key == pygame.K_r:
                teleport_to_initial_position()

    keys = pygame.key.get_pressed()
    run_mode = (
        keys[pygame.K_LSHIFT] or keys[pygame.K_RSHIFT]
    )  # Shift 키를 누르면 달리기 모드

    # 에이전트 이동 처리 (W, A, S, D)
    if keys[pygame.K_w]:
        move_agent("MoveAhead", run_mode)
    if keys[pygame.K_s]:
        move_agent("MoveBack", run_mode)
    if keys[pygame.K_a]:
        move_agent("MoveLeft", run_mode)
    if keys[pygame.K_d]:
        move_agent("MoveRight", run_mode)

    # 에이전트 회전 (Q, E)
    if keys[pygame.K_q]:
        controller.step(action="RotateLeft", degrees=rotate_step)
        if camera_mode == "third_person":
            update_third_person_camera()
        elif camera_mode == "top_down":
            update_top_down_camera()
    if keys[pygame.K_e]:
        controller.step(action="RotateRight", degrees=rotate_step)
        if camera_mode == "third_person":
            update_third_person_camera()
        elif camera_mode == "top_down":
            update_top_down_camera()

    # 카메라 회전 (I, K)
    if keys[pygame.K_i]:
        controller.step(action="LookUp", degrees=rotate_step)
    if keys[pygame.K_k]:
        controller.step(action="LookDown", degrees=rotate_step)

    # 이미지 렌더링
    if camera_mode == "egocentric":
        image = controller.last_event.frame
    else:
        image = controller.last_event.third_party_camera_frames[0]

    # 이미지 변환 (Pygame에서 사용할 수 있는 형식으로)
    image = np.rot90(image)
    surface = pygame.surfarray.make_surface(image)

    # 화면에 이미지 블릿
    screen.blit(surface, (0, 0))

    # 디스플레이 업데이트
    pygame.display.flip()

# Pygame 종료 및 AI2-THOR 컨트롤러 종료
pygame.quit()
controller.stop()

초기 위치와 회전이 저장되었습니다.
이동이 막혔습니다. 원하는 위치로 이동할 수 없습니다.
이동이 막혔습니다. 원하는 위치로 이동할 수 없습니다.
이동이 막혔습니다. 원하는 위치로 이동할 수 없습니다.
이동이 막혔습니다. 원하는 위치로 이동할 수 없습니다.
이동이 막혔습니다. 원하는 위치로 이동할 수 없습니다.
이동이 막혔습니다. 원하는 위치로 이동할 수 없습니다.
이동이 막혔습니다. 원하는 위치로 이동할 수 없습니다.
이동이 막혔습니다. 원하는 위치로 이동할 수 없습니다.
이동이 막혔습니다. 원하는 위치로 이동할 수 없습니다.
이동이 막혔습니다. 원하는 위치로 이동할 수 없습니다.
이동이 막혔습니다. 원하는 위치로 이동할 수 없습니다.
이동이 막혔습니다. 원하는 위치로 이동할 수 없습니다.
이동이 막혔습니다. 원하는 위치로 이동할 수 없습니다.
이동이 막혔습니다. 원하는 위치로 이동할 수 없습니다.
이동이 막혔습니다. 원하는 위치로 이동할 수 없습니다.
이동이 막혔습니다. 원하는 위치로 이동할 수 없습니다.
이동이 막혔습니다. 원하는 위치로 이동할 수 없습니다.
이동이 막혔습니다. 원하는 위치로 이동할 수 없습니다.
이동이 막혔습니다. 원하는 위치로 이동할 수 없습니다.
이동이 막혔습니다. 원하는 위치로 이동할 수 없습니다.
이동이 막혔습니다. 원하는 위치로 이동할 수 없습니다.
이동이 막혔습니다. 원하는 위치로 이동할 수 없습니다.
이동이 막혔습니다. 원하는 위치로 이동할 수 없습니다.
이동이 막혔습니다. 원하는 위치로 이동할 수 없습니다.
이동이 막혔습니다. 원하는 위치로 이동할 수 없습니다.
이동이 막혔습니다. 원하는 위치로 이동할 수 없습니다.
이동이 막혔습니다. 원하는 위치로 이동할 수 없습니다.
이동이 막혔습니다. 원하는 위치로 이동할 수 없습니다.
이동이 막혔습니다. 원하는 위치로 이동할 수 없습니다.
이동이 막혔습니다. 원하는 위치로 이동할 수 없습니다.
이동이 막혔습니다. 원하는 위치로 이동할 수 없습니다.
이동이 막혔습니다. 원하는 위치로 

In [7]:
# Pygame 종료 및 AI2-THOR 시뮬레이터 종료
pygame.quit()
controller.stop()